# MNQ Volume Climax Pullback - Risk Sizing Client Notebook

Ce notebook relit la campagne locale de refinement du sizing a risque fixe sur `MNQ`.

- alpha, signaux, horaires et exits restent ceux de la strategie VCP retenue,
- seul le sizing varie dans la grille locale autour de la zone gagnante,
- tous les parametres utiles du notebook sont regroupes dans une seule cellule,
- les graphiques principaux sont en Plotly pour une lecture client directe.


In [17]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Impossible de retrouver la racine du repo depuis le notebook.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


def fmt_money(value):
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value):,.1f} USD"


def fmt_pct(value, digits=1):
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value):.{digits}f}%"


def fmt_float(value, digits=3):
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value):.{digits}f}"


def resolve_variant_name(summary_frame, active_variant_name, risk_pct, max_contracts, skip_trade_if_too_small):
    if active_variant_name:
        match = summary_frame.loc[summary_frame["campaign_variant_name"].astype(str) == str(active_variant_name)].copy()
        if match.empty:
            raise ValueError(f"Variant {active_variant_name!r} not found in export.")
        return str(match.iloc[0]["campaign_variant_name"])

    mask = (
        pd.to_numeric(summary_frame["risk_pct"], errors="coerce").round(6).eq(round(float(risk_pct), 6))
        & pd.to_numeric(summary_frame["max_contracts"], errors="coerce").astype("Int64").eq(int(max_contracts))
        & pd.Series(summary_frame["skip_trade_if_too_small"], dtype="boolean").fillna(False).eq(bool(skip_trade_if_too_small))
    )
    match = summary_frame.loc[mask].copy()
    if match.empty:
        raise ValueError("No variant matches the requested notebook knobs.")
    ordered = match.sort_values(["oos_prop_score", "oos_net_pnl_usd"], ascending=[False, False]).reset_index(drop=True)
    return str(ordered.iloc[0]["campaign_variant_name"])


In [18]:
EXPORT_ROOT = ROOT / r"data\exports\volume_climax_pullback_mnq_risk_sizing_refinement_20260406_231223"

# Main notebook knobs
ACTIVE_VARIANT_NAME = None
ACTIVE_RISK_PCT = 0.0025
ACTIVE_MAX_CONTRACTS = 6
ACTIVE_SKIP_TRADE_IF_TOO_SMALL = True
COMPARE_BASELINE = True
COMPARE_PREVIOUS_WINNER = True
PLOT_TEMPLATE = "plotly_dark"

required_paths = {
    "summary_by_variant": EXPORT_ROOT / "summary_by_variant.csv",
    "summary_oos_only": EXPORT_ROOT / "summary_oos_only.csv",
    "trades_by_variant": EXPORT_ROOT / "trades_by_variant.csv",
    "daily_equity_by_variant": EXPORT_ROOT / "daily_equity_by_variant.csv",
    "prop_constraints_summary": EXPORT_ROOT / "prop_constraints_summary.csv",
    "heatmap_metrics": EXPORT_ROOT / "heatmap_metrics.csv",
    "robustness_zone_summary": EXPORT_ROOT / "robustness_zone_summary.csv",
    "run_metadata": EXPORT_ROOT / "run_metadata.json",
    "final_verdict": EXPORT_ROOT / "final_verdict.json",
    "final_report": EXPORT_ROOT / "final_report.md",
}

missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Fichiers manquants pour le notebook: {missing}")

print("EXPORT_ROOT =", EXPORT_ROOT)


EXPORT_ROOT = D:\Business\Trading\VSCODE\algo-trading-intraday-research\data\exports\volume_climax_pullback_mnq_risk_sizing_refinement_20260406_231223


In [19]:
run_metadata = json.loads((EXPORT_ROOT / "run_metadata.json").read_text(encoding="utf-8"))
final_verdict = json.loads((EXPORT_ROOT / "final_verdict.json").read_text(encoding="utf-8"))
final_report_text = (EXPORT_ROOT / "final_report.md").read_text(encoding="utf-8")

summary = pd.read_csv(EXPORT_ROOT / "summary_by_variant.csv")
summary_oos = pd.read_csv(EXPORT_ROOT / "summary_oos_only.csv")
trades = pd.read_csv(EXPORT_ROOT / "trades_by_variant.csv", parse_dates=["entry_time", "exit_time"], low_memory=False)
daily = pd.read_csv(EXPORT_ROOT / "daily_equity_by_variant.csv", parse_dates=["session_date"], low_memory=False)
prop_summary = pd.read_csv(EXPORT_ROOT / "prop_constraints_summary.csv")
heatmap_metrics = pd.read_csv(EXPORT_ROOT / "heatmap_metrics.csv")
cluster_summary = pd.read_csv(EXPORT_ROOT / "robustness_zone_summary.csv")

active_variant = resolve_variant_name(
    summary,
    active_variant_name=ACTIVE_VARIANT_NAME,
    risk_pct=ACTIVE_RISK_PCT,
    max_contracts=ACTIVE_MAX_CONTRACTS,
    skip_trade_if_too_small=ACTIVE_SKIP_TRADE_IF_TOO_SMALL,
)
baseline_variant = "fixed_1_contract"
previous_variant = final_verdict.get("recommended_variant") or final_verdict.get("punctual_oos_winner")
best_previous_tag = run_metadata.get("best_previous_winner_alias", "best_previous_winner")

variant_rows = {
    name: summary.loc[summary["campaign_variant_name"].astype(str) == str(name)].iloc[0]
    for name in {active_variant, baseline_variant, best_previous_tag, previous_variant}
    if summary["campaign_variant_name"].astype(str).eq(str(name)).any()
}

compare_names = [active_variant]
if COMPARE_BASELINE and baseline_variant not in compare_names:
    compare_names.append(baseline_variant)
if COMPARE_PREVIOUS_WINNER and best_previous_tag in variant_rows and best_previous_tag not in compare_names:
    compare_names.append(best_previous_tag)

display(Markdown(f"**Active variant:** `{active_variant}`"))
display(Markdown(f"**Final verdict:** `{final_verdict['final_verdict']}`"))


**Active variant:** `risk_pct_0p0025__max_contracts_6__skip_trade_if_too_small_true`

**Final verdict:** `retenir_une_zone_parametrique`

In [20]:
snapshot_cols = [
    "campaign_variant_name",
    "variant_role",
    "oos_net_pnl_usd",
    "oos_cagr_pct",
    "oos_sharpe",
    "oos_max_drawdown_usd",
    "oos_max_daily_drawdown_usd",
    "oos_prop_score",
    "oos_pass_target_3000_usd_without_breaching_2000_dd",
    "oos_avg_contracts",
    "oos_median_contracts",
    "oos_pct_trades_at_1_contract",
    "oos_pct_trades_at_2_contracts",
    "oos_pct_trades_at_3_plus_contracts",
    "oos_nb_skipped_trades",
    "oos_pct_skipped_trades",
]
display(Markdown("## 1. Snapshot OOS"))
display(summary.loc[summary["campaign_variant_name"].isin(compare_names), snapshot_cols].round(3))

if not cluster_summary.empty:
    display(Markdown("## 2. Zone robuste"))
    display(cluster_summary.round(4))


## 1. Snapshot OOS

,campaign_variant_name,variant_role,oos_net_pnl_usd,oos_cagr_pct,oos_sharpe,oos_max_drawdown_usd,oos_max_daily_drawdown_usd,oos_prop_score,oos_pass_target_3000_usd_without_breaching_2000_dd,oos_avg_contracts,oos_median_contracts,oos_pct_trades_at_1_contract,oos_pct_trades_at_2_contracts,oos_pct_trades_at_3_plus_contracts,oos_nb_skipped_trades,oos_pct_skipped_trades
0,fixed_1_contract,baseline,3038.575,2.891,1.451,790.0,790.00,1.189,True,1.000,1.0,100.000,0.000,0.000,0,0.000
15,risk_pct_0p0025__max_contracts_6__skip_trade_i...,grid,10426.075,9.582,1.656,732.0,742.60,6.638,True,3.088,2.0,37.363,13.187,49.451,10,9.901
31,best_previous_winner,best_previous_winner,6819.575,6.372,1.740,636.5,644.95,7.599,True,2.138,2.0,34.483,17.241,48.276,14,13.861


## 2. Zone robuste

,cluster_id,cluster_size,center_risk_pct,center_max_contracts,risk_pct_min,risk_pct_max,max_contracts_min,max_contracts_max,mean_oos_net_pnl_usd,mean_oos_sharpe,mean_oos_max_drawdown_usd,mean_oos_prop_score,best_variant_name,best_variant_prop_score,is_recommended_cluster
0,1,2,0.0015,4.5,0.0015,0.0015,4,5,6558.775,1.5326,516.25,7.9546,risk_pct_0p0015__max_contracts_4__skip_trade_i...,8.1581,True
1,2,1,0.0025,2.0,0.0025,0.0025,2,2,5651.975,1.8374,548.50,8.0828,risk_pct_0p0025__max_contracts_2__skip_trade_i...,8.0828,False


In [21]:
display(Markdown("## 3. Equity Curves"))

fig = make_subplots(
    rows=2,
    cols=2,
    shared_xaxes=False,
    vertical_spacing=0.10,
    horizontal_spacing=0.10,
    subplot_titles=("Full Sample Equity", "OOS Only Equity", "Full Sample Drawdown USD", "OOS Only Drawdown USD"),
)

colors = ["#16a34a", "#2563eb", "#f59e0b", "#7c3aed"]
for color, name in zip(colors, compare_names):
    full_curve = daily.loc[(daily["campaign_variant_name"] == name) & (daily["scope"] == "full")].copy()
    oos_curve = daily.loc[(daily["campaign_variant_name"] == name) & (daily["scope"] == "oos_only")].copy()
    fig.add_trace(go.Scatter(x=full_curve["session_date"], y=full_curve["equity"], mode="lines", name=f"{name} full", line=dict(color=color, width=2.5)), row=1, col=1)
    fig.add_trace(go.Scatter(x=oos_curve["session_date"], y=oos_curve["equity"], mode="lines", name=f"{name} oos", line=dict(color=color, width=2.5), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=full_curve["session_date"], y=full_curve["drawdown_usd"], mode="lines", name=f"{name} full dd", line=dict(color=color, width=1.6, dash="dot"), showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=oos_curve["session_date"], y=oos_curve["drawdown_usd"], mode="lines", name=f"{name} oos dd", line=dict(color=color, width=1.6, dash="dot"), showlegend=False), row=2, col=2)

fig.update_layout(template=PLOT_TEMPLATE, height=850, width=1500, legend=dict(orientation="h", y=-0.12))
fig.show()


## 3. Equity Curves

In [22]:
display(Markdown("## 4. Heatmaps OOS"))

metric_specs = [
    ("oos_net_pnl_usd", "OOS Net PnL", "RdYlGn"),
    ("oos_sharpe", "OOS Sharpe", "RdYlGn"),
    ("oos_max_drawdown_usd", "OOS MaxDD USD", "RdYlGn_r"),
    ("oos_prop_score", "OOS Prop Score", "RdYlGn"),
]

fig = make_subplots(rows=2, cols=2, subplot_titles=[title for _, title, _ in metric_specs], horizontal_spacing=0.10, vertical_spacing=0.14)
for index, (metric, title, colorscale) in enumerate(metric_specs, start=1):
    pivot = heatmap_metrics.pivot_table(index="risk_pct", columns="max_contracts", values=metric, aggfunc="mean").sort_index()
    row = 1 if index <= 2 else 2
    col = 1 if index in (1, 3) else 2
    fig.add_trace(
        go.Heatmap(
            z=pivot.values,
            x=[str(int(value)) for value in pivot.columns],
            y=[f"{float(value):.4f}" for value in pivot.index],
            colorscale=colorscale,
            text=pivot.round(2).astype(str).values,
            texttemplate="%{text}",
            hovertemplate="risk_pct=%{y}<br>max_contracts=%{x}<br>value=%{z}<extra></extra>",
        ),
        row=row,
        col=col,
    )

fig.update_layout(
    template=PLOT_TEMPLATE,
    height=900,
    width=1500,
)
fig.show()


## 4. Heatmaps OOS

In [23]:
display(Markdown("## 5. Active Variant Trade Profile"))

active_trades_oos = trades.loc[(trades["campaign_variant_name"] == active_variant) & (trades["scope"] == "oos_only")].copy()
baseline_trades_oos = trades.loc[(trades["campaign_variant_name"] == baseline_variant) & (trades["scope"] == "oos_only")].copy()

if not active_trades_oos.empty:
    quantity_mix = (
        active_trades_oos.assign(bucket=active_trades_oos["quantity"].map(lambda q: "3+" if q >= 3 else str(int(q))))
        .groupby("bucket", as_index=False)
        .agg(n_trades=("trade_id", "count"), net_pnl_usd=("net_pnl_usd", "sum"))
    )
    quantity_mix["pct_trades"] = quantity_mix["n_trades"] / quantity_mix["n_trades"].sum() * 100.0
    display(quantity_mix.round(2))

    mix_fig = px.bar(quantity_mix, x="bucket", y="pct_trades", text_auto=".1f", title="Active OOS position-size mix", labels={"bucket": "contracts bucket", "pct_trades": "% trades"})
    mix_fig.update_layout(template=PLOT_TEMPLATE, width=900, height=450)
    mix_fig.show()

hist_source = pd.concat(
    [
        baseline_trades_oos[["net_pnl_usd"]].assign(label="baseline"),
        active_trades_oos[["net_pnl_usd"]].assign(label="active"),
    ],
    ignore_index=True,
)
if not hist_source.empty:
    hist_fig = px.histogram(hist_source, x="net_pnl_usd", color="label", marginal="box", barmode="overlay", opacity=0.55, nbins=50, title="OOS trade PnL distribution")
    hist_fig.update_layout(template=PLOT_TEMPLATE, width=1100, height=500)
    hist_fig.show()


## 5. Active Variant Trade Profile

,bucket,n_trades,net_pnl_usd,pct_trades
0,1,34,909.43,37.36
1,2,12,2048.20,13.19
2,3+,45,7468.45,49.45


In [24]:
display(Markdown("## 6. Final Readout"))
display(Markdown(final_report_text))


## 6. Final Readout

# Volume Climax Pullback MNQ Risk Sizing - Refinement Report

## Scope
- Symbol: `MNQ` only.
- Base alpha reused unchanged: `dynamic_exit_atr_target_1p0_ts2_vq0p95_bf0p5_ra1p2`.
- Reference V3 run: `D:\Business\Trading\VSCODE\algo-trading-intraday-research\data\exports\volume_climax_pullback_v3_run\volume_climax_pullback_v3_20260402_184702`.
- Dataset: `repo latest MNQ source`.
- Sessions: full `1747` | IS `1222` | OOS `525`.
- Sizing logic unchanged from the prior campaign. Only the local grid around the previous winner was refined.

## Prop Score
- Formula on OOS: `prop_score = 4 * min(net_pnl/3000, 2) + 3 * min(sharpe, 2.5) - 3 * min(maxDD/2000, 3) - 5 * min(max_daily_DD/1000, 3) - 2 * min(worst_trade/200, 3) - 0.5 * nb_days_below_-500 - 6 * 1[pass=False]`.
- Interpretation: reward enough OOS profit to clear a 50k target, reward clean Sharpe, penalize drawdown, penalize daily drawdown strongly, and penalize any non-pass configuration heavily.

## OOS Winner
- Punctual OOS winner by `prop_score`: `risk_pct_0p0015__max_contracts_4__skip_trade_if_too_small_true`.
- OOS metrics: net `6151.67` | CAGR `5.77%` | Sharpe `1.541` | maxDD `497.50` | max daily DD `571.50` | prop_score `8.16`.
- Versus previous winner tag: net `-667.90` | Sharpe `-0.199` | maxDD `-139.00`.

```text
                                         campaign_variant_name  oos_net_pnl_usd  oos_sharpe  oos_max_drawdown_usd  oos_max_daily_drawdown_usd  oos_prop_score  oos_pass_target_3000_usd_without_breaching_2000_dd
risk_pct_0p0015__max_contracts_4__skip_trade_if_too_small_true         6151.675    1.540608               497.500                     571.500        8.158074                                                True
risk_pct_0p0025__max_contracts_2__skip_trade_if_too_small_true         5651.975    1.837363               548.500                     548.500        8.082805                                                True
risk_pct_0p0015__max_contracts_5__skip_trade_if_too_small_true         6965.875    1.524516               535.000                     609.000        7.751047                                                True
risk_pct_0p0025__max_contracts_3__skip_trade_if_too_small_true         6819.575    1.739649               636.500                     644.950        7.599446                                                True
risk_pct_0p0015__max_contracts_6__skip_trade_if_too_small_true         7770.075    1.497854               581.725                     655.725        7.352351                                                True
risk_pct_0p0025__max_contracts_4__skip_trade_if_too_small_true         7920.800    1.691914               653.000                     707.000        7.041243                                                True
risk_pct_0p0015__max_contracts_3__skip_trade_if_too_small_true         4962.200    1.542505               474.175                     548.175        6.941643                                                True
risk_pct_0p0025__max_contracts_5__skip_trade_if_too_small_true         9357.600    1.687010               732.000                     738.550        6.750280                                                True
risk_pct_0p0025__max_contracts_6__skip_trade_if_too_small_true        10426.075    1.656324               732.000                     742.600        6.637972                                                True
risk_pct_0p0020__max_contracts_6__skip_trade_if_too_small_true         9409.550    1.577245               725.325                     799.325        6.447123                                                True
```

## Robust Zone
- Recommended cluster id: `1` | size `2` | scale `zone_etruite_mais_non_isolee`.
- Zone range: risk_pct `0.0015` -> `0.0015` | max_contracts `4` -> `5`.
- Zone center: risk_pct `0.0015` | max_contracts `4.50`.
- Zone means: net `6558.77` | Sharpe `1.533` | maxDD `516.25` | prop_score `7.95`.
- Best point inside the zone: `risk_pct_0p0015__max_contracts_4__skip_trade_if_too_small_true`.

## Requested Readout
1. Le gagnant ponctuel OOS: `risk_pct_0p0015__max_contracts_4__skip_trade_if_too_small_true`.
2. La zone robuste OOS: cluster `1` couvrant risk_pct `0.0015` -> `0.0015` et cap `4` -> `5`.
3. Impact marginal de risk_pct autour de 0.25%: voir la coupe `max_contracts=3` ci-dessous. La question utile est de savoir si le score reste propre quand on s'eloigne de `0.0025`.
4. Impact marginal du cap `max_contracts` autour de 3: voir la coupe `risk_pct=0.0025` ci-dessous. La lecture utile est la vitesse de deterioration du drawdown et du `prop_score` quand on desserre le cap.
5. Distribution des tailles pour la meilleure variante: avg `2.36` | median `2.00` | 1c `41.8%` | 2c `14.9%` | 3+c `43.3%`.
6. Trades skippes pour la meilleure variante: `35` soit `34.3%` des tentatives.
7. Robustesse: `zone_etruite_mais_non_isolee`.
8. Reference de recherche: `risk_pct_0p0015__max_contracts_4__skip_trade_if_too_small_true` | mode prop-safe: `risk_pct_0p0015__max_contracts_4__skip_trade_if_too_small_true` | mode plus agressif mais defendable: `risk_pct_0p0025__max_contracts_6__skip_trade_if_too_small_true`.
9. Verdict final: `retenir_une_zone_parametrique`.

### Slice: risk_pct around 0.25% with max_contracts=3
```text
                                         campaign_variant_name  risk_pct  max_contracts  oos_net_pnl_usd  oos_sharpe  oos_max_drawdown_usd  oos_max_daily_drawdown_usd  oos_prop_score
risk_pct_0p0015__max_contracts_3__skip_trade_if_too_small_true    0.0015            3.0         4962.200    1.542505               474.175                     548.175        6.941643
risk_pct_0p0020__max_contracts_3__skip_trade_if_too_small_true    0.0020            3.0         5553.500    1.601962               732.675                     806.675        5.938166
risk_pct_0p0025__max_contracts_3__skip_trade_if_too_small_true    0.0025            3.0         6819.575    1.739649               636.500                     644.950        7.599446
risk_pct_0p0030__max_contracts_3__skip_trade_if_too_small_true    0.0030            3.0         6792.475    1.616171               946.500                     946.500        5.076264
risk_pct_0p0035__max_contracts_3__skip_trade_if_too_small_true    0.0035            3.0         7622.900    1.663280               955.475                     985.975        4.586751
risk_pct_0p0040__max_contracts_3__skip_trade_if_too_small_true    0.0040            3.0         8232.975    1.691477              1100.500                    1100.500        3.761180
```

### Slice: max_contracts around 3 with risk_pct=0.0025
```text
                                         campaign_variant_name  risk_pct  max_contracts  oos_net_pnl_usd  oos_sharpe  oos_max_drawdown_usd  oos_max_daily_drawdown_usd  oos_prop_score
risk_pct_0p0025__max_contracts_2__skip_trade_if_too_small_true    0.0025            2.0         5651.975    1.837363                 548.5                      548.50        8.082805
risk_pct_0p0025__max_contracts_3__skip_trade_if_too_small_true    0.0025            3.0         6819.575    1.739649                 636.5                      644.95        7.599446
risk_pct_0p0025__max_contracts_4__skip_trade_if_too_small_true    0.0025            4.0         7920.800    1.691914                 653.0                      707.00        7.041243
risk_pct_0p0025__max_contracts_5__skip_trade_if_too_small_true    0.0025            5.0         9357.600    1.687010                 732.0                      738.55        6.750280
risk_pct_0p0025__max_contracts_6__skip_trade_if_too_small_true    0.0025            6.0        10426.075    1.656324                 732.0                      742.60        6.637972
```
